# NB00 — One-off ETL

**Question:** Turn raw WRDS earnings-call zips and yfinance prices into the four canonical tables every downstream notebook reads from `data/clean/`.

## Pipeline

```
earningcall/*.zip ──┐
                    ├──▶ NB00 (this notebook):
yfinance API ───────┘
   1. Filter transcripts — match 50 target tickers via NAME_TO_TICKER, keep Q1–Q4 YYYY only
                            → data/clean/transcripts_us/{TICKER}_{YEAR}_Q{Q}__*.json
   2. Build datasets     — flatten per-call JSONs into session/segment tables
                            → data/clean/earnings_sessions.{csv,parquet}
                            → data/clean/earnings_segments.{csv,parquet}
   3. Fetch prices       — yfinance for 50 tickers + SPY benchmark
                            → data/clean/prices.parquet
   4. Build labels       — 10-day excess-of-SPY return → binary up/down label
                            → data/clean/labels.parquet
   5. Verify             — file sizes, row counts, positive rate
```

**Run order:** Section 1 → 2 → 3 → 4. Each section is idempotent (re-running overwrites outputs).

**Status:** all four steps have already produced their outputs in `data/clean/`. You only need to re-run if you changed `config.py` (e.g. ticker list, time range).

## Output schema (the contract for NB01–NB05)

| File | Grain | Used by |
|------|-------|---------|
| `earnings_sessions.parquet` | one row per (ticker, year, quarter) call | NB01–NB05 |
| `earnings_segments.parquet` | one row per speaker turn | NB01, NB02 |
| `prices.parquet` | one row per (ticker, date), 51 tickers incl. SPY | NB01, NB03, NB04, NB05 |
| `labels.parquet` | one row per call, binary `label = 1[excess_return_10d > 0]` | NB01, NB04 (auxiliary) |


In [ ]:
import sys, json, re, time, zipfile, csv as csv_module
from pathlib import Path
from collections import defaultdict
import pandas as pd

# Make project root importable
sys.path.insert(0, str(Path('..').resolve()))
import config
from src.llm_agent.price_data import fetch_prices, compute_excess_return
from src.llm_agent.data_loader import load_sessions, load_prices

config.CLEAN_DIR.mkdir(parents=True, exist_ok=True)
config.TRANSCRIPTS_DIR.mkdir(parents=True, exist_ok=True)
print('PROJECT_ROOT:', config.PROJECT_ROOT)
print('TICKERS     :', len(config.TICKERS), 'companies')

## 1. Filter transcripts (zip → per-call JSON)

Iterates `earningcall/*.zip`, keeps only quarterly earnings calls (`Q1–Q4 YYYY` in the headline) for the 50 target tickers, and writes one JSON per call to `data/clean/transcripts_us/`.

In [ ]:
QUARTER_RE = re.compile(r'\bQ([1-4])\s+(\d{4})\b', re.IGNORECASE)

def match_ticker(company_name: str):
    if not company_name:
        return None
    cn = company_name.lower()
    for pat, tk in config.NAME_TO_TICKER.items():
        if pat in cn:
            return tk
    return None

def extract_year_quarter(headline: str):
    if not headline:
        return None, None
    m = QUARTER_RE.search(headline)
    return (int(m.group(2)), int(m.group(1))) if m else (None, None)

def process_zip(zip_path, dst, stats, coverage):
    try:
        zf = zipfile.ZipFile(zip_path, 'r')
    except Exception as e:
        print(f'  cannot open {zip_path.name}: {e}')
        return
    for name in (n for n in zf.namelist() if n.endswith('.json')):
        stats['scanned'] += 1
        try:
            obj = json.loads(zf.read(name).decode('utf-8', errors='replace'))
        except Exception:
            stats['skipped_parse'] += 1
            continue
        for r in (obj if isinstance(obj, list) else [obj]):
            ticker = match_ticker(r.get('companyname') or '')
            if not ticker:
                stats['skipped_wrong_company'] += 1; continue
            year, quarter = extract_year_quarter(r.get('headline') or '')
            if year is None:
                stats['skipped_non_quarterly'] += 1; continue
            if not (config.START_YEAR <= year <= config.END_YEAR):
                stats['skipped_out_of_range'] += 1; continue
            out_name = f'{ticker}_{year}_Q{quarter}__{Path(name).name}'
            (dst / out_name).write_bytes(json.dumps(r, ensure_ascii=False).encode('utf-8'))
            coverage[ticker].add((year, quarter))
            stats['matched'] += 1
    zf.close()

ZIP_DIR = config.PROJECT_ROOT / 'earningcall'   # local zip directory
DST = config.TRANSCRIPTS_DIR
zips = sorted(ZIP_DIR.glob('*.zip'))
print(f'Found {len(zips)} zips, target dir: {DST}')

stats = defaultdict(int); coverage = defaultdict(set)
t0 = time.time()
for i, zp in enumerate(zips, 1):
    process_zip(zp, DST, stats, coverage)
    print(f'[{i:>2}/{len(zips)}] {zp.name}  matched_total={stats["matched"]}')
print(f'\nDone in {time.time()-t0:.0f}s.  Matched: {stats["matched"]}')
print(f'Skipped — wrong company: {stats["skipped_wrong_company"]}, '
      f'non-quarterly: {stats["skipped_non_quarterly"]}, '
      f'out-of-range: {stats["skipped_out_of_range"]}, '
      f'parse-fail: {stats["skipped_parse"]}')

## 2. Build datasets (JSON → session/segment tables)

Flattens the per-call JSONs into two tables:
- `earnings_sessions.csv` — one row per call (overview)
- `earnings_segments.parquet` — one row per speaker turn (used by the LLM/NLP pipeline)

In [ ]:
def parse_filename(filename: str):
    try:
        ticker, year, quarter = filename.split('__')[0].split('_')
        return ticker, int(year), int(quarter[1:])
    except Exception:
        return None, None, None

files = sorted(config.TRANSCRIPTS_DIR.glob('*.json'))
print(f'Scanning {len(files)} JSON files...')

sessions_rows, segments_rows = [], []
for fp in files:
    ticker, year, quarter = parse_filename(fp.name)
    if ticker is None:
        continue
    rec = json.loads(fp.read_text(encoding='utf-8'))
    components = rec.get('components') or []

    full_text_parts, type_counts = [], {}
    for c in components:
        sp, co, ty = c.get('personname') or '', c.get('companyofperson') or '', c.get('componenttypename') or ''
        tx = c.get('text') or ''
        full_text_parts.append(f'[{sp} | {co} | {ty}]\n{tx}' if sp else tx)
        type_counts[ty or 'Unknown'] = type_counts.get(ty or 'Unknown', 0) + 1
    full_text = '\n\n'.join(full_text_parts)

    sessions_rows.append({
        'ticker': ticker, 'fiscal_year': year, 'fiscal_quarter': quarter,
        'companyname': rec.get('companyname'), 'companyid': rec.get('companyid'),
        'transcriptid': rec.get('transcriptid'), 'keydevid': rec.get('keydevid'),
        'headline': rec.get('headline'),
        'mostimportantdate': rec.get('mostimportantdate'),
        'transcriptcreationdate': rec.get('transcriptcreationdate'),
        'n_components': len(components),
        'component_types': json.dumps(type_counts, ensure_ascii=False),
        'n_chars': len(full_text), 'n_words_est': len(full_text.split()),
        'full_text': full_text,
    })
    for order_idx, c in enumerate(components):
        segments_rows.append({
            'ticker': ticker, 'fiscal_year': year, 'fiscal_quarter': quarter,
            'transcriptid': rec.get('transcriptid'),
            'mostimportantdate': rec.get('mostimportantdate'),
            'componentid': c.get('componentid'),
            'componentorder': c.get('componentorder', order_idx),
            'componenttypename': c.get('componenttypename'),
            'personname': c.get('personname'),
            'companyofperson': c.get('companyofperson'),
            'text': c.get('text'), 'n_chars': len(c.get('text') or ''),
        })

df_sessions = pd.DataFrame(sessions_rows).sort_values(['ticker','fiscal_year','fiscal_quarter']).reset_index(drop=True)
df_segments = pd.DataFrame(segments_rows).sort_values(['ticker','fiscal_year','fiscal_quarter','componentorder']).reset_index(drop=True)

df_sessions.to_csv(config.CLEAN_DIR / 'earnings_sessions.csv',
                   index=False, quoting=csv_module.QUOTE_NONNUMERIC, encoding='utf-8')
df_segments.to_parquet(config.CLEAN_DIR / 'earnings_segments.parquet',
                       index=False, compression='zstd')
df_segments.to_csv(config.CLEAN_DIR / 'earnings_segments.csv',
                   index=False, quoting=csv_module.QUOTE_NONNUMERIC, encoding='utf-8')
print(f'sessions: {len(df_sessions)}  segments: {len(df_segments)}')
print('Type distribution:'); print(df_segments['componenttypename'].value_counts().head(10))

## 3. Fetch prices (yfinance → `prices.parquet`)

In [ ]:
tickers = config.TICKERS + [config.BENCHMARK]
print(f'Fetching {len(tickers)} tickers, {config.PRICE_START_DATE} to {config.PRICE_END_DATE}')
df_px = fetch_prices(tickers, config.PRICE_START_DATE, config.PRICE_END_DATE)
out = config.CLEAN_DIR / 'prices.parquet'
df_px.to_parquet(out, index=False, compression='zstd')
print(f'\nSaved {len(df_px)} rows  ({df_px.ticker.nunique()} tickers)  to {out}')
print(f'Date range: {df_px.date.min()}  to  {df_px.date.max()}')

## 4. Build labels (10-day excess return → `labels.parquet`)

In [ ]:
sessions = load_sessions()
prices = load_prices()
prices['date'] = pd.to_datetime(prices['date'])

rows = []
for _, r in sessions.iterrows():
    excess = compute_excess_return(
        prices, r['ticker'], config.BENCHMARK,
        from_date=r['mostimportantdate'],
        holding_days=config.LABEL_HOLDING_DAYS,
    )
    rows.append({
        'ticker': r['ticker'], 'fiscal_year': r['fiscal_year'],
        'fiscal_quarter': r['fiscal_quarter'], 'transcriptid': r['transcriptid'],
        'event_date': r['mostimportantdate'],
        'excess_return_10d': excess,
        'label': int(excess > 0) if excess is not None else None,
    })

df_lab = pd.DataFrame(rows)
df_lab.to_parquet(config.CLEAN_DIR / 'labels.parquet', index=False, compression='zstd')
print(f'Valid labels: {df_lab.label.notna().sum()} / {len(df_lab)}')
print(f'Positive rate: {df_lab.label.mean():.1%}')

## Verify outputs

In [ ]:
for fp in sorted(config.CLEAN_DIR.glob('*')):
    if fp.is_file():
        print(f'{fp.name:<35s}  {fp.stat().st_size/1024/1024:>7.1f} MB')
    else:
        n = sum(1 for _ in fp.iterdir())
        print(f'{fp.name:<35s}  {n} files')